<a href="https://colab.research.google.com/github/shaktidharreddy/Agentic_AI_eduhubspot_aug_sep_2026/blob/main/Module00_Python_OpenAI/OpenAI_API_StructuredOutput_with_Streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
from google.colab import userdata

# Retrieve the OpenAI API key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [18]:
# first example using pydantic

import os
from openai import OpenAI
from pydantic import BaseModel

# Instantiate client explicitly passing the environment variable or letting it load natively
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# 📝 Define the exact schema blueprint
class ProductDetails(BaseModel):
    item: str
    price: int

# Fixed: Using the dedicated '.parse()' endpoint to handle Pydantic objects properly
completion = client.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "I bought a vintage leather jacket for 150 dollars"}
    ],
    # 🛠️ Activates Structured Outputs via response_format
    response_format=ProductDetails
)

# print(completion.choices[0].message.content)

# Print structured response
print(completion.choices[0].message.parsed)

print(completion.choices[0].message.parsed.item)

print(completion.choices[0].message.parsed.price)

item='Vintage Leather Jacket' price=150
Vintage Leather Jacket
150


In [6]:
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI()

# 1. Define your blueprint
class ProductDetails(BaseModel):
    item: str
    price: int

# 2. Use the dedicated .parse() method
response = client.responses.parse(
    model="gpt-6-luna",
    input="I bought a vintage leather jacket for 150 dollars.",
    # Passes the class directly using text_format
    text_format=ProductDetails
)

# 3. Access your fully validated and parsed Pydantic object
parsed_data = response.output_parsed

print(parsed_data.item)   # Output: vintage leather jacket
print(parsed_data.price)  # Output: 150
print(type(parsed_data))  # Output: <class '__main__.ProductDetails'>

Vintage leather jacket
150
<class '__main__.ProductDetails'>


In [20]:
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI()

# 1. Define your structured output schema
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

# 2. Stream using client.responses.stream() and text_format
with client.responses.stream(
    model="gpt-6-luna",
    input="Extract the event(while extractingg strictly to adhere to the facts in the sentence, dont add anything from your pre-trained knwoeldge): Alice and Bob are going to a science fair on Friday.",
    text_format=CalendarEvent,
) as stream:
    for event in stream:
        # Print raw JSON token deltas as they arrive
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)

    # 3. Get the fully validated Pydantic object once the stream finishes
    final_response = stream.get_final_response()
    if final_response.status == "completed":
        parsed_event: CalendarEvent = final_response.output_parsed
        print("\n\nParsed Object:", parsed_event)


{"name":"science fair","date":"Friday","participants":["Alice","Bob"]}

Parsed Object: name='science fair' date='Friday' participants=['Alice', 'Bob']


In [21]:
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI()

# 1. Define your structured output schema
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

# 2. Stream using .stream() and response_format
with client.chat.completions.stream(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {"role": "user", "content": "Alice and Bob are going to a science fair on Friday."},
    ],
    response_format=CalendarEvent,
) as stream:
    for event in stream:
        # Print raw JSON token deltas as they arrive
        if event.type == "content.delta":
            print(event.delta, end="", flush=True)

        # Optionally grab the parsed object as soon as content finishes
        elif event.type == "content.done":
            print("\n\nParsed Object:", event.parsed)

    # 3. Or retrieve the full completion object after the loop
    final_completion = stream.get_final_completion()
    parsed_event: CalendarEvent = final_completion.choices[0].message.parsed


{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}

Parsed Object: name='Science Fair' date='Friday' participants=['Alice', 'Bob']


In [24]:
import json
from openai import OpenAI

client = OpenAI()

# Fixed: Filled out cut-off property strings, types, and booleans
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "product_schema",
        "strict": True,  # Required for Structured Outputs
        "schema": {
            "type": "object",
            "properties": {
                "item": {"type": "string"},
                "price": {"type": "integer"}
            },
            "required": ["item", "price"],
            "additionalProperties": False
        }
    }
}

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "I bought a vintage leather jacket for 150 dollars."}
    ],
    response_format=response_format
)

# Raw string output must be manually parsed into a python dictionary
raw_json_string = completion.choices[0].message.content
print(raw_json_string)
parsed_data = json.loads(raw_json_string)

print(type(parsed_data))

print(parsed_data["item"])   # Output: vintage leather jacket
print(parsed_data["price"])  # Output: 150


{"item":"Vintage Leather Jacket","price":150}
<class 'dict'>
Vintage Leather Jacket
150


In [29]:

import json
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant. You must always respond with a valid JSON object."
        },
        {
            "role": "user",
            "content": "List 3 common programming languages and their primary use case."
        }
    ],
    # Enforces that the model's output is valid JSON syntax
    response_format={"type": "json_object"}
)

# Extract the raw JSON string
raw_json_string = response.choices[0].message.content
print("Raw Output:\n", raw_json_string)

# Parse it into a standard Python dictionary
parsed_data = json.loads(raw_json_string)
print("\nParsed Dictionary:", parsed_data)

Raw Output:
 {
  "programming_languages": [
    {
      "name": "Python",
      "primary_use_case": "General-purpose programming, commonly used for web development, data analysis, artificial intelligence, and scientific computing."
    },
    {
      "name": "JavaScript",
      "primary_use_case": "Web development, primarily used for creating interactive and dynamic front-end web applications."
    },
    {
      "name": "Java",
      "primary_use_case": "Enterprise-level applications, mobile app development (especially Android), and large system back-ends."
    }
  ]
}

Parsed Dictionary: {'programming_languages': [{'name': 'Python', 'primary_use_case': 'General-purpose programming, commonly used for web development, data analysis, artificial intelligence, and scientific computing.'}, {'name': 'JavaScript', 'primary_use_case': 'Web development, primarily used for creating interactive and dynamic front-end web applications.'}, {'name': 'Java', 'primary_use_case': 'Enterprise-level appl